# Fine Tuning a local LLM with QLoRA to add tool use capability on a non-tool using base model
## Overview
This is a small project to demonstrate LLM fine-tuning techniques, tool usage, quantization, and evaluations.
## Motivation
Most of the modern LLMs have the capability to use tools, but there are still many old, smaller, or open-source models that do not have this capability. This project aims to bridge that gap by allowing users to add tool use capabilities to any of their favorite models using QLoRA.
## Author
- Name: Isai Roberto Sotarriva Alvarez
- GitHub: [https://github.com/irsotarriva](https://github.com/irsotarriva)
- linkedin: [https://www.linkedin.com/in/irsotarriva](https://www.linkedin.com/in/irsotarriva)
## Licence
This project is licensed under the [Apache-2.0 License](https://www.apache.org/licenses/LICENSE-2.0), the models and datasets used may have their own respective licenses, please read and comply with them accordingly.
## Libraries Used
- PyTorch
- Transformers
- peft
- Datasets
- BitsAndBytes
- Hugging Face Hub
- lighteval
## Datasets
- [Hugging Face roborovski/synthetic-tool-calls-v2](https://huggingface.co/datasets/roborovski/synthetic-tool-calls-v2)
- [Hugging Face HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk)
## Models
- Base Model: [Hugging Face HuggingFaceTB/smolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B)
- Reference Model: [Hugging Face HuggingFaceTB/smolLM2-1.7B-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct)
## Important papers to keep in mind
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761)
- [SmolLM2: When Smol Goes Big -- Data-Centric Training of a Small Language Model](https://arxiv.org/abs/2502.02737v1)


# Starting point
Let's start by downloading the base model and the reference model from Hugging Face.
On this project we will use the [HuggingFaceTB/smolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B) as the base model and the [HuggingFaceTB/smolLM2-1.7B-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct) as a comparison for what the fully fine-tuned model capabilities should be like after the QLoRA fine-tuning.
We will be using the [Hugging Face HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) as the main dataset for the fine-tuning, since we want to comapre our work with the official instruct fine-tuned model from the smolLM2 team which was trained on this exact dataset to add instruction following capabilities to the base model.

In [1]:
#!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu121
!pip install vllm-tpu
from vllm import LLM, SamplingParams
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)
llm = LLM(model="HuggingFaceTB/SmolLM2-1.7B")
outputs = llm.generate(prompts, sampling_params=sampling_params)
for i, output in enumerate(outputs):
    print(f"Prompt {i+1}: {prompts[i]}")
    print(f"Completion: {output.text}\n")
#launch as a vLLM server (equivalent to the cli vllm serve MODEL_NAME)
class VllmServer:
    def __init__(self, model_name, port, tokenizer_path=None):
        self.model_name = model_name
        self.port = port
        self.tokenizer_path = tokenizer_path

    def launch(self):
        llm = LLM(model=self.model_name, tokenizer_path=self.tokenizer_path)
        llm.serve(port=self.port)

RuntimeError: Only one platform plugin can be activated, but got: ['cuda', 'tpu']

We will do a quick benchmark of both the base model and the instruct fine-tuned model using [lighteval](https://github.com/huggingface/lighteval)
to ensure we are being able to reproduce the results shown in the [SmolLM2 hugging Face model card](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B#base-pre-trained-model) and the [SmolLM2 instruct hugging Face model card](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct#instruct-fine-tuned-model) respectively.

# Fine-tuning with QLoRA to add instruction following capabilities
We will be using the [Hugging Face HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) dataset to fine-tune the base model using QLoRA to add instruction following capabilities.
Unlike the official instruct fine-tuned model which was trained by doing supervised fine-tuning on the whole model, we will be using QLoRA to do low-rank adaptation on the quantized base model to achieve similar results with a fraction of the resources.
We will be using the peft library to implement QLoRA and the bitsandbytes library to handle the 8-bit quantization.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

# Load the dataset from Hugging Face
ds = load_dataset("HuggingFaceTB/smoltalk", "all")
# Define a function to preprocess the dataset
def preprocessData(example):
    messages = example["messages"]
    inputText = ""
    for message in messages:
        inputText += f"{message['role']}: {message['content']}\n"
    return {"input_text": inputText.strip()}

# Preprocess the dataset
ds = ds.map(preprocessData, remove_columns=["messages"])

# Define a data collator for batching
dataCollator = DataCollatorForSeq2Seq(tokenizer, model=base_model)

# Create DataLoaders for train and test splits
trainLoader = DataLoader(
    ds["train"], batch_size=8, shuffle=True, collate_fn=dataCollator
)
testLoader = DataLoader(
    ds["test"], batch_size=8, shuffle=False, collate_fn=dataCollator
)